In [ ]:
# Import libraries
import pandas as pd
import numpy as np

In [ ]:
# Load dataset
df = pd.read_excel("it_jobs.xlsx")

In [ ]:
# Preview data
df.head()

In [ ]:
# Keep only the relevant columns for analysis
columns_to_keep = [
    "title",
    "job_type",
    "interval",
    "is_remote",
    "job_level",
    "mean_salary",
    "cleaned_description",
]

In [ ]:
df = df[columns_to_keep].copy()

In [ ]:
# Create placeholder column for experience (to be filled later)
df["experience_years"] = np.nan

In [ ]:
# Quick check
df.head()

In [ ]:
df.info()
df.describe(include="all")

In [ ]:
import re

In [ ]:
def infer_job_type(desc):
    desc = desc.lower()
    if re.search(r"full[- ]?time", desc):
        return "fulltime"
    elif re.search(r"part[- ]?time", desc):
        return "parttime"
    elif re.search(r"\bcontract\b", desc):
        return "contract"
    elif re.search(r"intern(ship)?", desc):
        return "internship"
    elif re.search(r"temporary", desc):
        return "temporary"
    elif re.search(r"freelance", desc):
        return "freelance"
    return None

In [ ]:
# Apply inference only where job_type is missing
df["job_type"] = df.apply(
    lambda row: (
        infer_job_type(row["cleaned_description"])
        if pd.isnull(row["job_type"])
        else row["job_type"]
    ),
    axis=1,
)

In [ ]:
# Function to detect if the job is remote based on the cleaned description
def is_remote_job(desc):
    desc = str(desc).lower()
    keywords = [
        "remote",
        "work from home",
        "telecommute",
        "fully remote",
        "work remotely",
    ]
    return int(any(keyword in desc for keyword in keywords))

In [ ]:
# Apply the function to fill missing values in 'is_remote' with 1 (True) or 0 (False)
df.loc[df["is_remote"].isna(), "is_remote"] = (
    df.loc[df["is_remote"].isna(), "cleaned_description"]
    .apply(is_remote_job)
    .astype(int)
)

In [ ]:
df["is_remote"] = df["is_remote"].astype(int)

In [ ]:
def classify_education_type(text):
    if pd.isna(text) or text.strip() == "":
        return "Online/Courses"

    text = text.lower()

    # University-related keywords including degree types
    university_keywords = [
        "university",
        "college",
        "institute",
        "school",
        "academy",
        "polytechnic",
        "bachelor",
        "master",
        "msc",
        "phd",
        "doctor",
        "ba",
        "bs",
        "ma",
        "mba",
        "b.sc",
        "m.sc",
    ]

    # Known Bootcamps
    bootcamp_keywords = [
        "bootcamp",
        "hack reactor",
        "general assembly",
        "code institute",
        "le wagon",
        "ironhack",
        "springboard",
        "flatiron",
    ]

    # Online course platforms
    online_keywords = ["coursera", "udemy", "udacity", "edx", "freecodecamp", "online"]

    # High School / Secondary
    highschool_keywords = [
        "high school",
        "secondary school",
        "lycee",
        "baccalaureate",
        "senior school",
        "gcse",
    ]

    if any(k in text for k in bootcamp_keywords):
        return "Bootcamp"
    elif any(k in text for k in highschool_keywords):
        return "High School"
    elif any(k in text for k in online_keywords):
        return "Online/Courses"
    elif any(k in text for k in university_keywords):
        return "University"
    else:
        return "Online/Courses"

In [ ]:
# Apply to the correct column
df["Education_Type"] = df["cleaned_description"].apply(classify_education_type)

In [ ]:
def extract_experience_years(text):
    if pd.isnull(text):
        return np.nan  # Keep missing as NaN

    text = text.lower()

    # Explicitly handle "no experience" or "zero experience"
    if re.search(r"\b(no|zero)\s+experience\b", text):
        return 0

    # Exclude misleading age phrases
    age_phrases = ["years old", "year old", "years of age", "age of", "over the past"]
    if any(phrase in text for phrase in age_phrases):
        return np.nan  # Consider these as unknown (not experience)

    # Experience patterns
    patterns = [
        r"(\d{1,2})\s*-\s*(\d{1,2})\s*years.*experience",
        r"minimum of (\d{1,2})\s*years",
        r"at least (\d{1,2})\s*years",
        r"(\d{1,2})\s*or more\s*years",
        r"(\d{1,2})\+?\s*years?\s*(?:of)?\s*(?:experience|exp|working)?",
        r"(?:experience|exp)[^\d]{0,10}(\d{1,2})\+?\s*years?",
    ]

    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            try:
                if len(match.groups()) == 2:
                    value = int(match.group(1))  # lower bound of range
                else:
                    value = int(match.group(1))
                if value <= 20:
                    return value
            except Exception:
                return np.nan

    # No match found — return NaN (unknown)
    return np.nan

In [ ]:
# Apply to your dataframe
df["experience_years"] = df["cleaned_description"].apply(extract_experience_years)

In [ ]:
# Set salaries below 1000 USD to NaN (mark as missing)
df.loc[df["mean_salary"] < 1000, "mean_salary"] = np.nan

In [ ]:
# Fill missing 'is_remote' values with 0 (assume not remote)
df["is_remote"] = df["is_remote"].fillna(0).astype(int)

In [ ]:
# Ensure 'experience_years' column is numeric
df["experience_years"] = pd.to_numeric(df["experience_years"], errors="coerce")

In [ ]:
# Define job level based on experience
def classify_job_level_by_years(years):
    if pd.isna(years):
        return None
    if years < 2:
        return "entry_level"
    elif 2 <= years < 5:
        return "mid_level"
    elif 5 <= years < 10:
        return "senior_level"
    else:
        return "executive_level"

In [ ]:
# Apply classification
df["job_level"] = df["experience_years"].apply(classify_job_level_by_years)

In [ ]:
required_cols = ["job_type", "job_level", "mean_salary", "experience_years"]
df = df.dropna(subset=required_cols).reset_index(drop=True)

In [ ]:
def categorize_job_title(title):
    title = str(title).lower()

    # IT Support / Helpdesk
    if any(
        keyword in title
        for keyword in [
            "support",
            "help desk",
            "technician",
            "service desk",
            "desktop support",
            "it support",
        ]
    ):
        return "IT Support / Helpdesk"

    # Network & Systems
    elif any(
        keyword in title
        for keyword in [
            "network",
            "systems administrator",
            "system administrator",
            "systems engineer",
            "network engineer",
            "infrastructure",
            "cloud engineer",
        ]
    ):
        return "Network & Systems"

    # Cybersecurity
    elif any(
        keyword in title
        for keyword in [
            "cybersecurity",
            "security",
            "devsecops",
            "cyber security",
            "security analyst",
            "security architect",
            "security engineer",
            "infosec",
        ]
    ):
        return "Cybersecurity"

    # Software Development
    elif any(
        keyword in title
        for keyword in [
            "developer",
            "software engineer",
            "full-stack",
            "frontend",
            "backend",
            "web developer",
            "app developer",
            "java",
            "python",
            "programmer",
        ]
    ):
        return "Software Development"

    # Project / Program Management
    elif any(
        keyword in title
        for keyword in [
            "project manager",
            "program manager",
            "scrum master",
            "agile",
            "pmo",
            "project coordinator",
        ]
    ):
        return "Project / Program Management"

    # Data & Analytics
    elif any(
        keyword in title
        for keyword in [
            "data",
            "data scientist",
            "data analyst",
            "business analyst",
            "bi analyst",
            "analytics",
            "machine learning",
            "ml engineer",
            "ai",
        ]
    ):
        return "Data & Analytics"

    # IT Management / Director
    elif any(
        keyword in title
        for keyword in [
            "director",
            "manager",
            "vp",
            "vice president",
            "head",
            "chief",
            "cto",
            "cio",
        ]
    ):
        return "IT Management / Director"

    # Consulting / Specialist
    elif any(
        keyword in title
        for keyword in [
            "consultant",
            "specialist",
            "solutions architect",
            "enterprise architect",
            "technical architect",
        ]
    ):
        return "Consulting / Specialist"

    # Intern / Entry Level
    elif any(
        keyword in title
        for keyword in ["intern", "junior", "entry", "graduate trainee"]
    ):
        return "Intern / Entry Level"

    # Check if it's IT-related before putting in Other
    elif any(
        keyword in title
        for keyword in [
            "IT Project Engineer",
            "it",
            "IT",
            "engineer",
            "computer",
            "technology",
            "systems",
            "software",
            "hardware",
            "developer",
            "network",
            "cyber",
            "programmer",
            "data",
            "cloud",
            "security",
            "tech",
            "digital",
        ]
    ):
        return "General IT / Other IT Role"

    else:
        return "Other / Miscellaneous"

In [ ]:
df["job_category"] = df["title"].apply(categorize_job_title)

In [ ]:
# Drop the column to finalize the dataset
df.drop(columns=["cleaned_description"], inplace=True)
print("🧹 'cleaned_description' column removed from dataset.")

In [ ]:
# Save the cleaned DataFrame to a CSV file
df.to_csv("cleaned_it_jobs_data.csv", index=False)
print("✅ Cleaned data saved successfully as it_jobs_cleaned.csv")